# Explore generation experiments

This notebook walks through the four helpers in `json_storm.analysis`, using our
Qwen3.6 and GPT-6.1-Sol sweeps from 3 November 2026. We start with one experiment,
inspect its batches and candidates, then compare prompts and models.

Everything here is read-only: no model calls, MongoDB queries or experiment changes.
Accepted queries have passed parsing, target checks and persistence—not MongoDB execution.

## 1. Environment and experiment selection

Run `uv sync --locked` from the repository root and select its `.venv` Python kernel.
These examples require the original experiments in your local Yanex store
(default `~/.yanex/experiments`). On another machine, first copy/sync those experiments
or replace the IDs below. Explicit IDs keep our comparison reproducible.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import yanex.results as yr
from IPython.display import display

from json_storm.analysis import (
    generation_batches_table,
    generation_candidates_table,
    generation_runs_table,
    summarize_generation_run,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 90)

# Each list is ordered p1 through p7.
sol_ids = ["57d6f1c3", "8e4ea30b", "6654acdc", "d78c69d7", "52a02a82", "af97d240", "9190cd17"]
qwen_ids = ["28b9755c", "669c89c4", "8adf1c41", "cf3ae5f7", "5f3c7b72", "4ef16c01", "e8999d55"]

sol = [yr.get_experiment(run_id) for run_id in sol_ids]
qwen = [yr.get_experiment(run_id) for run_id in qwen_ids]

experiment = sol[5]  # p6 contains our one persistence failure.
print(type(experiment).__name__, experiment.id, experiment.name, experiment.status)
print("Script:", experiment.script_path)

## 2. `summarize_generation_run`: one experiment

The helper accepts the native Yanex `Experiment` and returns a dictionary. We turn
it into a one-row table for display. All four helpers require script basename
`generate.py`, so passing a schema experiment raises `ValueError`.

Yields are fractions: 0.95 means 95%. Missing measurements stay missing; they are
not zero. Metrics come from the latest logged step, falling back to the checkpoint
manifest only if no metrics were logged.


In [ ]:
summary = summarize_generation_run(experiment)
summary_columns = [
    "experiment_id",
    "prompt_id",
    "model",
    "status",
    "metrics_source",
    "num_queries",
    "queries_parsed",
    "queries_accepted",
    "queries_persistence_failed",
    "accepted_yield",
    "unique_queries",
    "distinct_yield",
    "model_seconds",
]
display(pd.DataFrame([summary])[summary_columns].round(3))

## 3. `generation_batches_table`: one row per model call

Sol p6 has four decoded responses. A decoded batch can still contain rejected
candidates, so batch success and candidate acceptance are different measurements.
Elapsed time is in milliseconds; usage fields are normalized from saved responses.


In [ ]:
batches = generation_batches_table(experiment)
display(
    batches[
        [
            "batch_index",
            "requested_count",
            "outcome",
            "finish_reason",
            "elapsed_ms",
            "input_tokens",
            "output_tokens",
            "reasoning_tokens",
        ]
    ].round(1)
)

For contrast, Qwen p1 has three undecodable batches. Their requested slots are
known, but the number of actual candidates inside those malformed responses is not.
The error objects below preserve the original parser diagnostics.


In [ ]:
qwen_batches = generation_batches_table(qwen[0])
display(qwen_batches[["batch_index", "requested_count", "outcome", "parse_error"]])

## 4. `generation_candidates_table`: inspect individual outcomes

Each row represents one decoded candidate, identified by zero-based batch index
and position. No rows are invented for undecodable batches. Accepted candidates
also have a workload position and BONDI fingerprint.


In [ ]:
candidates = generation_candidates_table(experiment)
display(
    candidates[
        [
            "batch_index",
            "position",
            "outcome",
            "accepted",
            "workload_position",
            "fingerprint",
        ]
    ].head(8)
)
display(candidates.groupby("outcome").size().rename("candidates").to_frame())

rejected = candidates.loc[~candidates["accepted"].eq(True)]
display(rejected[["batch_index", "position", "error", "persistence_error"]])
# Show the original projection behind this failure without rewriting the query.
display(rejected.iloc[0]["raw"]["projection"])

This p6 candidate parsed, but a BSON regex inside its projection could not be
serialized by the installed BONDI version. Its raw query and error remain available.

Fingerprints also let us inspect repeated queries. Sol p4 has 20 accepted entries
but only 14 distinct fingerprints. All repeated occurrences are retained below;
we do not silently remove them from the measured yield.


In [ ]:
p4_candidates = generation_candidates_table(sol[3])
accepted = p4_candidates.loc[p4_candidates["accepted"].eq(True)].dropna(subset=["fingerprint"])
duplicates = accepted.loc[accepted.duplicated("fingerprint", keep=False)]
display(
    duplicates[["batch_index", "position", "workload_position", "fingerprint"]].sort_values(
        ["fingerprint", "workload_position"]
    )
)

## 5. `generation_runs_table`: compare seven prompts

This helper stacks summaries without averaging repetitions or excluding failures.
First inspect the Sol sweep on its own. Distinct yield measures exact fingerprint
uniqueness within each run; it is not a measure of semantic diversity.


In [ ]:
sol_runs = generation_runs_table(sol)
display(
    sol_runs[
        [
            "prompt_id",
            "queries_accepted",
            "unique_queries",
            "exact_duplicate_queries",
            "accepted_yield",
            "distinct_yield",
            "model_seconds",
            "output_tokens_per_accepted",
        ]
    ].round(3)
)

## 6. Compare models for each prompt

Ordinary pandas operations are enough to build comparisons. Keep configuration
columns visible before interpreting differences. Both sweeps request 20 queries
with batch size 5 and the same schema. Reasoning settings and inference routes differ:
Sol uses low reasoning via subscription; Qwen runs locally with thinking off.


In [ ]:
runs = generation_runs_table(sol + qwen)
display(
    runs[
        [
            "experiment_id",
            "prompt_id",
            "model",
            "batch_size",
            "num_queries",
            "target_collection",
            "model_parameters",
            "schema_sha256",
        ]
    ]
)
print("Distinct rendered schemas:", runs["schema_sha256"].nunique())

# Short labels are only for display; the original model column stays intact.
runs["model_label"] = runs["model"].map(
    {
        "chatgpt/gpt-6.1-sol": "Sol (low reasoning)",
        "ollama_chat/qwen3.6:latest": "Qwen (thinking off)",
    }
)
yield_by_prompt = runs.pivot(index="prompt_id", columns="model_label", values="accepted_yield")
display((100 * yield_by_prompt).round(1).rename_axis(columns=None))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
(100 * yield_by_prompt).plot.bar(ax=ax, rot=0, color=["#a9b6c6", "#247a92"])
ax.set(
    title="Accepted yield: same schema, 20 queries, batch size 5",
    xlabel="Prompt",
    ylabel="Accepted / requested (%)",
    ylim=(0, 110),
)
ax.legend(title=None, loc="lower right")
fig.tight_layout()
plt.show()

## 7. Compare one prompt across models

Filter first when your question concerns one prompt. Change `selected_prompt` to
explore another. The same pattern works for batch-size studies: select one prompt
and model, then sort by `batch_size`. These two sweeps have only batch size 5, so
they cannot tell us the effect of changing batch size.


In [ ]:
selected_prompt = "p1"
single_prompt = runs.loc[runs["prompt_id"].eq(selected_prompt)]
display(
    single_prompt[
        [
            "model_label",
            "batch_size",
            "queries_accepted",
            "unique_queries",
            "model_seconds",
            "seconds_per_accepted",
            "output_tokens_per_accepted",
            "output_tokens_reported_calls",
            "batches_attempted",
        ]
    ].round(2)
)

## 8. Interpret the comparison

- Sol accepted 139/140 queries; Qwen accepted 86/140. Acceptance still does not prove
  that MongoDB can execute a query or that it returns documents.
- Sol has repetitions in p3/p4, which acceptance alone hides.
- Time per accepted query includes time spent on failed responses. Token ratios
  are only as complete as the reported-call coverage; providers use different tokenizers.
- One run per prompt is descriptive evidence, not a replicated model ranking.
- `pivot` intentionally raises if you later select multiple runs per model/prompt.
  Keep those replicate IDs and choose an explicit aggregation rather than silently
  averaging them. Pooled yield and mean run yield answer different questions.

For interrupted runs, an empty candidate table can mean either an empty artifact
or a missing one: inspect `table.attrs["artifact_available"]`. Batch tables mark
missing artifacts per row. Call `experiment.refresh()` before rereading a live run.

Next, try changing the experiment selection and the columns displayed. Plotting and
saving reports remain notebook responsibilities; the four helpers only read data.
